In [1]:
import torch

In [2]:
# 1.构造输入数据 x : (100个样本, 3个特征)
x = torch.randn(100, 3)
# 2.构造权重 w : (3个特征, 1个输出)
#预设真实的权重和偏置（我们要让模型无限接近这两个数)
true_w = torch.tensor([[2.0], [-1.0], [0.5]])
true_b = 0.5

# 3. 生成带噪声的标签 y: (100, 1)
#y_true = x @ true_w + true_b + 0.1 * torch.randn(100, 1) # @ 是矩阵乘法
y_true = torch.matmul(x, true_w) + true_b + 0.1 * torch.randn(100, 1)
# 这里触发了广播：(100, 3) @ (3, 1) -> (100, 1)，再加上标量 0.5

初始化我们要训练的参数
这是你 GitHub 模板的起点。注意 requires_grad=True。

In [3]:
# 初始化W为随机值，b为0
# W 形状必须是 (3, 1) 才能和 x (100, 3) 做矩阵乘法
w = torch.randn(3,1,requires_grad=True) # 需要计算梯度
# 标量偏置，待会儿靠广播加到每一行
b = torch.zeros(1, requires_grad=True)   # 需要计算梯度

训练循环

In [5]:
learning_rate = 0.1

for epoch in range(100):
    # --- 1. 前向传播 (Forward) ---
    # 这里的矩阵乘法和广播是核心
    # x: (100, 3) @ w: (3, 1) -> (100, 1)
    # 然后加上 b (1,)，触发广播，b 会加到这 100 行的每一个结果上
    y_pred=torch.matmul(x,w)+b

    loss = torch.mean((y_pred-y_true)**2)

    if w.grad is not None:
        w.grad.zero_()
        b.grad.zero_()

    loss.backward()

    with torch.no_grad():
        w-=learning_rate*w.grad
        b-=learning_rate*b.grad

    if epoch % 10==0:
        print(f"Epoch{epoch},Loss:{loss.item():.4f}",f"w:{w.squeeze().tolist()}",f"b:{b.item():.4f}")


Epoch0,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch10,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch20,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch30,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch40,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch50,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch60,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch70,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch80,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165
Epoch90,Loss:0.0089 w:[1.9981459379196167, -0.9992371201515198, 0.5112414360046387] b:0.5165


In [ ]:
import torch

# 1. 数据准备 (保持简单的函数形式)
def create_linear_data(num_samples=100):
    x = torch.randn(num_samples, 3)
    true_w = torch.tensor([[2.0], [-1.0], [0.5]])
    true_b = 0.5
    # y = x @ w + b + 噪声
    y_true = torch.matmul(x, true_w) + true_b + 0.1 * torch.randn(num_samples, 1)
    return x, y_true

# 2. 手写线性回归类 (将参数 w, b 封装在一起)
class LinearRegression:
    def __init__(self, in_features):
        # 权重初始化为随机，偏置初始化为 0
        self.w = torch.randn(in_features, 1, requires_grad=True)
        self.b = torch.zeros(1, requires_grad=True)
        
        # 将参数存入列表，方便后续循环处理
        self.params = [self.w, self.b]

    def forward(self, x):
        # 核心逻辑：y = xW + b
        return torch.matmul(x, self.w) + self.b

# 3. 训练配置
X, y_true = create_linear_data(100)
model = LinearRegression(in_features=3)
learning_rate = 0.1
epochs = 100

# 4. 训练循环 (所有核心逻辑都在这里，一目了然)
for epoch in range(epochs):
    # --- 前向传播 ---
    y_pred = model.forward(X)
    
    # --- 计算 MSE Loss ---
    loss = torch.mean((y_pred - y_true)**2)
    
    # --- 梯度清零 ---
    for p in model.params:
        if p.grad is not None:
            p.grad.zero_()
            
    # --- 反向传播 ---
    loss.backward()
    
    # --- 参数更新 ---
    with torch.no_grad():
        for p in model.params:
            p -= learning_rate * p.grad
            
    # --- 日志打印 ---
    if (epoch + 1) % 10 == 0:
        # 将 w 展平方便打印查看结果
        w_val = model.w.squeeze().tolist()
        print(f"Epoch [{epoch+1:>3}/{epochs}] | Loss: {loss.item():.4f} | w: {[round(i, 4) for i in w_val]} | b: {model.b.item():.4f}")

print("\n训练完成！")